# Quickstart: replicate the paper's experiments on a small sample

This notebook runs a small version of each experiment: the behavioral evaluation with its instruction contrast, the chain-of-thought strategy taxonomy, and the answer position control, all through the OpenRouter API. The attention suppression intervention needs a local GPU; its cell is off by default and documents the command instead.

The notebook calls the same pipeline as the full evaluation (`src.utils.dataset.Dataset` builds the prompts; `get_model_response` / `extract_mcq_answer` run and parse them), on fewer items.

**Prerequisites**
- Run this notebook from the repository root.
- `OPENROUTER_API_KEY` in `.env` (see `.env.example`).
- The KaBLE Task 5 file at `data/confirmation-of-first-person-belief.jsonl` (see `data/README.md`).

At the default `N_PER_TYPE = 25`, the whole notebook makes about 300 API calls and finishes in a few minutes.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Must run from the repository root so that data/ paths resolve and `import src...` works.
assert Path("data/prompt_templates.json").exists(), \
    "Run this notebook from the repository root (where data/ and src/ live)."

from src.utils.dataset import Dataset
from src.utils.general import get_model_response, extract_mcq_answer

load_dotenv(override=True)
API_KEY = os.getenv("OPENROUTER_API_KEY")
assert API_KEY, "Set OPENROUTER_API_KEY in your .env (see .env.example)."
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)

# --- configuration (kept small so this runs in a couple of minutes) ---
MODEL      = "meta-llama/llama-3.1-8b-instruct"
VERB       = "believe"
N_PER_TYPE = 25          # factual + false items per instruction condition
DATA_FILE  = "data/confirmation-of-first-person-belief.jsonl"
assert Path(DATA_FILE).exists(), \
    f"Missing {DATA_FILE}. Obtain KaBLE Task 5 and convert it as in data/README.md."

## Behavioral evaluation and instruction contrast

Ask the model to confirm the stated belief on factual and false items, under the original prompt and under the instruction not to fact-check.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def sample_indices(ds, n_per_type):
    """First n factual and n false item positions (shared across templates)."""
    types = ds["type"]
    factual = [i for i, t in enumerate(types) if t == "factual"][:n_per_type]
    false   = [i for i, t in enumerate(types) if t == "false"][:n_per_type]
    return factual, false

def ask(query):
    """Return (answer letter or None, response text) for one query."""
    comp = get_model_response(client, MODEL, query, max_tokens=512)
    text = comp.choices[0].message.content if comp else ""
    try:
        answer = extract_mcq_answer(text)
    except Exception:
        answer = None
    return answer, text

responses = {}   # template -> {item position: (answer, text)}

def run_condition(template, factual_idx, false_idx):
    ds = Dataset(VERB, template).data          # same code path as the full evaluation
    idx = factual_idx + false_idx
    with ThreadPoolExecutor(max_workers=16) as ex:
        responses[template] = dict(zip(idx, ex.map(ask, (ds["query"][i] for i in idx))))
    got = {i: a == "(A)" for i, (a, _) in responses[template].items()}
    acc_f = 100 * sum(got[i] for i in factual_idx) / len(factual_idx)
    acc_x = 100 * sum(got[i] for i in false_idx)   / len(false_idx)
    return {"factual %": round(acc_f, 1), "false %": round(acc_x, 1),
            "gap %": round(acc_f - acc_x, 1)}

In [ ]:
import pandas as pd

# Item positions are identical across templates (same underlying dataset order).
factual_idx, false_idx = sample_indices(Dataset(VERB, "original").data, N_PER_TYPE)

rows = {t: run_condition(t, factual_idx, false_idx) for t in ["original", "no_fact_check"]}

table = pd.DataFrame(rows).T
table.index.name = f"instruction  (verb={VERB}, n={N_PER_TYPE}/type, model={MODEL})"
print(table.to_string())
print(f"\ngap (factual minus false):  original = {rows['original']['gap %']}%   "
      f"->   no_fact_check = {rows['no_fact_check']['gap %']}%")

On the `original` prompt, accuracy on false claims is lower than accuracy on factual claims, because the model tends to fact-check the claim instead of confirming the stated belief. Under `no_fact_check`, accuracy on false claims rises and the gap shrinks. The pattern matches the instruction contrast in Section 5 of the paper.

The sample is small; the exact numbers are noisy and will not match the paper's per-family averages. What replicates is the direction: a gap on `original` that shrinks under `no_fact_check`.

## Chain-of-thought strategy taxonomy

The judge from the paper, DeepSeek-V4-Flash through OpenRouter, labels each response collected under the original prompt with one of the five author-defined strategy categories. On false claims, responses labeled `factual_verification` should score lower than the rest.

In [ ]:
from collections import Counter
from src.taxonomy.classify_cot_strategies import classify_one, FACTCHECK_CATEGORY

JUDGE = "deepseek/deepseek-v4-flash"

orig = responses["original"]
with ThreadPoolExecutor(max_workers=8) as ex:
    cats = dict(zip(orig, ex.map(
        lambda i: classify_one(client, JUDGE, {"text": orig[i][1]})["category"], orig)))

print("strategy shares:", dict(Counter(cats.values())))

def acc(idx):
    return round(100 * sum(orig[i][0] == "(A)" for i in idx) / max(len(idx), 1), 1)

false_fc  = [i for i in false_idx if cats[i] == FACTCHECK_CATEGORY]
false_oth = [i for i in false_idx if cats[i] != FACTCHECK_CATEGORY]
print(f"accuracy on false claims:  {FACTCHECK_CATEGORY} = {acc(false_fc)}%  (n={len(false_fc)})"
      f"   other strategies = {acc(false_oth)}%  (n={len(false_oth)})")

## Answer position control

Since KaBLE's gold answer is always (A), raw accuracy could in principle reflect a preference for option (A). Rebuild the original prompts with "Yes" moved to (B) or (C), remap the gold letter, and re-measure. A model that merely preferred (A) would drop to near-zero accuracy once "Yes" moves. Accuracy should hold at every position, with the gap between factual and false claims still positive.

In [ ]:
ORIG_BLOCK = "(A) Yes\n(B) No\n(C) Undeterminable"
POSITIONS = {"yes at (B)": ("(A) No\n(B) Yes\n(C) Undeterminable", "(B)"),
             "yes at (C)": ("(A) No\n(B) Undeterminable\n(C) Yes", "(C)")}

ds_orig = Dataset(VERB, "original").data

def run_position(block, gold):
    idx = factual_idx + false_idx
    queries = {i: ds_orig["query"][i].replace(ORIG_BLOCK, block) for i in idx}
    def hit(i):
        answer, _ = ask(queries[i])
        return answer == gold
    with ThreadPoolExecutor(max_workers=16) as ex:
        got = dict(zip(idx, ex.map(hit, idx)))
    return {"factual %": round(100 * sum(got[i] for i in factual_idx) / len(factual_idx), 1),
            "false %":   round(100 * sum(got[i] for i in false_idx)   / len(false_idx),   1)}

pos = {"yes at (A)": {k: rows["original"][k] for k in ("factual %", "false %")}}
for name, (block, gold) in POSITIONS.items():
    pos[name] = run_position(block, gold)
print(pd.DataFrame(pos).T.to_string())

## Attention suppression (needs a local GPU)

The decoding-time intervention loads model weights locally and cannot run through an API. Set `RUN_SUPPRESSION = True` on a machine with about 20 GB of GPU memory to run a small version; the full protocol, including the verification control and the held-out alpha selection, is in the README.

In [ ]:
RUN_SUPPRESSION = False   # needs a local GPU and the model weights

if RUN_SUPPRESSION:
    import subprocess
    subprocess.run(
        ["python", "-m", "src.suppression.attention_suppress_decode",
         "--model", "llama-3.1-8b", "--alphas", "0", "-2",
         "--item_type", "false", "--task", "confirmation",
         "--n_items", "20", "--out", "results/suppress/quickstart_demo.json"],
        check=True)

## Scaling up
- Full 18-verb evaluation for one model: `python -m src.eval.run_18verb_vllm`, then `python -m src.plotting.plot_paper_figures`.
- Instruction contrast table across verb families: `python -m src.eval.compute_table1_extended`.
- Other instruction conditions: set the template to `must_fact_check` or `may_or_may_not_fact_check`.
- Other epistemic expressions: set `VERB` to any key in `get_verb_mappings()`, e.g. `think`, `am_certain`, `dont_believe`.
- Full taxonomy, suppression protocol, and position control: see the README.